In [1]:
import numpy as np
from sklearn.externals.array_api_extra.testing import override
from model_wrapper import *
import cuml.accel
from math import floor
cuml.accel.install()

# of Training Instances: 47
# of Testing Instances: 11
Current RAM usage: 298.43 MB


In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.decomposition import IncrementalPCA
from sklearn.kernel_approximation import RBFSampler

class SGDModel(Model):
    def __init__(self, anatomical_plane, fluid_sensitive=None, fat_suppression=None, pca_n_comp=20, rbf_n_comp=5, rbf_training_size=20):
        self.pca_n_comp = pca_n_comp
        self.rbf_n_comp = rbf_n_comp
        self.rbf_training_size = rbf_training_size
        self.inc_pca = None
        self.rbf_sampler = None
        self.model = OneVsRestClassifier(SGDClassifier(loss="log_loss", random_state=42))
        super().__init__(anatomical_plane, fluid_sensitive, fat_suppression, full_train=False)

    @override
    def batch_fit(self, training_folders: pd.Series, y: np.ndarray):
        if len(training_folders) <= self.pca_n_comp:
            self.inc_pca = IncrementalPCA(n_components=len(training_folders))
            n_batches = 1
        else:
            self.inc_pca = IncrementalPCA(n_components=self.pca_n_comp)
            n_batches = floor(len(training_folders) / self.pca_n_comp)

        self.rbf_sampler = RBFSampler(gamma=0.1, n_components=min(len(training_folders), self.rbf_n_comp), random_state=42)
        img_count = training_folders.apply(get_img_count)
        training_folders = training_folders[img_count >= MIN_IMG_COUNT]

        for folders_batch in np.array_split(training_folders, n_batches):
            X_batch = np.array([get_training_instance(f) for f in folders_batch])
            X_batch = np.reshape(X_batch, shape=(X_batch.shape[0], -1))
            self.inc_pca.partial_fit(X_batch)

        self.rbf_n_comp = min(self.rbf_n_comp, len(training_folders))
        self.rbf_training_size = min(self.rbf_training_size, len(training_folders))
        rbf_training_batch = np.array([get_training_instance(training_folders[i]) for i in range(self.rbf_training_size)])
        rbf_training_batch = np.reshape(
            rbf_training_batch,
            shape=(rbf_training_batch.shape[0], -1)
        )
        rbf_training_batch = self.inc_pca.transform(rbf_training_batch)
        self.rbf_sampler.fit(rbf_training_batch)

        batch_size = self.pca_n_comp
        for start_idx in range(0, len(training_folders), batch_size):
            end_idx = start_idx + batch_size
            folders_batch = training_folders[start_idx:end_idx]
            y_batch = y[start_idx:end_idx]

            X_batch = np.array([get_training_instance(f) for f in folders_batch])
            X_batch = np.reshape(X_batch, (X_batch.shape[0], -1))
            X_batch = self.inc_pca.transform(X_batch)
            X_batch = self.rbf_sampler.transform(X_batch)

            self.model.partial_fit(X_batch, y_batch)

    @override
    def predict_batch(self, x: np.ndarray) -> np.ndarray:
        x = np.reshape(x, shape=(x.shape[0], -1))
        x_reduced = self.inc_pca.transform(x)
        x_rbf = self.rbf_sampler.transform(x_reduced)
        return self.model.predict(x_rbf)

    @override
    def predict_instance(self, x: np.ndarray) -> np.ndarray:
        x = np.reshape(x, shape=(1, -1))
        x_reduced = self.inc_pca.transform(x)
        x_rbf = self.rbf_sampler.transform(x_reduced)
        pred_ = self.model.predict(x_rbf)
        return np.reshape(pred_, shape=(pred_.shape[1]))